# Vectorización de Documentos hacia Supabase

### Requisitos previos antes de ejecutar este notebook

* Crear en supabase:
    - Tabla que soporta almacenamiento de vectores en supabase (usa la extensión `pgvector`): `documentos_vectorizados`
    - Función que realiza la búsqueda de similitud vectorial en supabase: `similar_documentos_vectorizados`.

        Esto se logra ejecutando (en un proyecto de trabajo de supabase) la query del archivo `snippet_para_Supabase.sql` (Ver: https://supabase.com/docs/guides/ai/langchain?queryGroups=database-method&database-method=sql)

* Credenciales de Supabase:
    - SUPABASE_URL
    - SUPABASE_SECRET_KEY

---

## Contexto: ¿Qué es RAG?

**RAG** (*Retrieval-Augmented Generation*) es una arquitectura que extiende las capacidades de un LLM incorporando **recuperación de información** de una base de conocimientos externa antes de generar cada respuesta. Esto permite:

- Responder sobre información actualizada o privada (no incluida en el entrenamiento del modelo)
- Reducir alucinaciones al anclar la respuesta en documentos reales
- Mantener la fuente de verdad separada del modelo, facilitando su actualización

## Pipeline de este notebook

Este notebook cubre la **Fase 1** del sistema RAG: ingestar, dividir y vectorizar los documentos fuente para almacenarlos en Supabase (PostgreSQL con `pgvector`).

```mermaid
flowchart LR
    PDF["PDF\nDocumento fuente"]
    L["1 — Document\nLoader"]
    S["2 — Text\nSplitter\nChunks"]
    E["3 — Embedding\nModel\nVectores"]
    DB[("Supabase\npgvector")]

    PDF --> L --> S --> E --> DB
```

| Paso | Qué hace | Herramienta |
|------|----------|-------------|
| 1 | Carga el PDF y extrae el texto página por página | `pypdf.PdfReader` + `langchain_core.documents.Document` |
| 2 | Divide el texto en fragmentos con superposición | `RecursiveCharacterTextSplitter` |
| 3 | Convierte cada chunk en un vector numérico | `OpenAIEmbeddings` — `text-embedding-ada-002` (1536 dims) |
| 4 | Configura la conexión a Supabase | `supabase-py` |
| 5 | Sube vectores + texto a la tabla en Supabase | `SupabaseVectorStore.from_documents()` |

In [1]:
#!uv add langchain_community langchain_text_splitters supabase pypdf

In [2]:
import os

#Paso 1: Elección de la Técnica de DocumentLoader
from pypdf import PdfReader
from langchain_core.documents import Document

#Paso 2: Elección de Técnica de Splitting
from langchain_text_splitters import RecursiveCharacterTextSplitter #Mi técnica de Splitting

#Paso 3: Elección del Modelo de Word Embedding
from langchain_openai import OpenAIEmbeddings

#Importanciones para trabajar con SUPABASE
from langchain_community.vectorstores import SupabaseVectorStore
from supabase import create_client

/tmp/ipykernel_27274/3254231755.py:14: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import SupabaseVectorStore


In [3]:
# ===========================================
# Se asegura que se carguen las variables de entorno desde el archivo .env
# ===========================================
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv())

True

## Paso 1 — Document Loader

El primer paso es leer el documento fuente y convertirlo en una lista de objetos `Document` de LangChain. Cada objeto contiene:

- `page_content`: el texto extraído de la página
- `metadata`: información adicional como número de página y ruta del archivo

Se usa `pypdf.PdfReader` directamente (en lugar de `PyPDFLoader` de `langchain_community`, que está siendo descontinuado) junto con `langchain_core.documents.Document` para construir los objetos con el mismo formato que esperan los componentes de LangChain en los pasos siguientes.

In [4]:
#=================================== Paso 1: Document Loader =======================================
path = "Base_de_Conocimientos/BROCHURE Bootcamp AI ENGINEER FOR DEVELOPERS.pdf"
reader = PdfReader(path)
documentos = [
    Document(page_content=page.extract_text(), metadata={"source": path, "page": i})
    for i, page in enumerate(reader.pages)
]
documentos

[Document(metadata={'source': 'Base_de_Conocimientos/BROCHURE Bootcamp AI ENGINEER FOR DEVELOPERS.pdf', 'page': 0}, page_content=' B o o t c a m p  \nAI ENGINEER FORDEVELOPERS\nDomina el desarrollo de soluciones inteligentes con LLMs,\nRAG y agentes multiagente. Aprende a diseñar, integrar,\nmonitorear y desplegar arquitecturas de IA generativa\nen entornos cloud, low-code y tiempo real.\nHERRAMIENTAS\nOpenAI Google AI Anthropic Meta AI Mistral AI Docker DeepSeek LangChain\nLangGraph LlamaIndex CrewAI FastAPI Qdrant Supabase Firestore Evolution\nAPI\nChatwoot Streamlit LangFuse LiveKit Deepgram V0'),
 Document(metadata={'source': 'Base_de_Conocimientos/BROCHURE Bootcamp AI ENGINEER FOR DEVELOPERS.pdf', 'page': 1}, page_content='¿Sabías que hoy los desarrolladores pueden construir agentes inteligentes\ncapaces de razonar, colaborar entre sí y desplegarse en entornos\nproductivos con seguridad y escalabilidad?\nCon esa visión nace el Bootcamp de Inteligencia Artificial para Developers,\n

In [5]:
print(f"El documento contiene {len(documentos)} páginas.")

El documento contiene 11 páginas.


## Paso 2 — Text Splitting (Chunking)

Los LLMs tienen un límite en su ventana de contexto, por lo que no es posible enviar documentos completos. El proceso de **chunking** divide el texto en fragmentos manejables que luego se vectorizan individualmente.

`RecursiveCharacterTextSplitter` divide el texto respetando primero separadores semánticos naturales (párrafos → saltos de línea → oraciones → palabras) antes de cortar por caracteres. Esto preserva la coherencia del fragmento mejor que un corte fijo por longitud.

A continuación se detallan las consideraciones para elegir `chunk_size` y `chunk_overlap`:

### Recomendaciones para elegir el "chunk_size" y "chunk_overlap" en la técnica de Splitting:

*   chunk_size:
    Tamaño del fragmento de texto que se va a crear. 
    - Un tamaño más grande puede capturar más contexto, pero también puede ser más costoso en términos de recursos y tiempo de procesamiento.
    - Un tamaño más pequeño puede ser más eficiente, pero puede perder contexto importante.

* chunk_overlap:
    Cantidad de superposición entre fragmentos consecutivos, lo que permite que el LLM pueda mantener el contexto.
    - Una superposición mayor puede ayudar a capturar mejor el contexto, pero también puede aumentar la redundancia y el tamaño  vectorstore.

Se recomienda que el tamaño de los chunks sea una potencia de 2, para que el modelo de embeddings pueda procesarlos de manera más eficiente. 


In [6]:
# Se recomienda que el tamaño de los chunks sea una potencia de 2, para que el modelo de embeddings pueda procesarlos de manera más eficiente. 
# Por ejemplo, si se desea un tamaño de chunk de 512 caracteres, se puede establecer n=9, ya que 2^9 = 512.
n_chars_for_chunks = 2**9

n_chars_for_overlap = 2**8

print(f"Se usarán chunks de {n_chars_for_chunks} caracteres con un overlap de {n_chars_for_overlap} caracteres.")

Se usarán chunks de 512 caracteres con un overlap de 256 caracteres.


In [7]:
#======================================= Paso 2: Chunking ===========================================
text_splitter =  RecursiveCharacterTextSplitter(
    chunk_size = n_chars_for_chunks,
    chunk_overlap = n_chars_for_overlap,
)

chunks = text_splitter.split_documents(
    documents=documentos
)

print(f"Se generaron {len(chunks)} chunks")

Se generaron 46 chunks


In [8]:
for i, chunk in enumerate(chunks):
    print(f"-"*50)
    print(f"Chunk {i+1}: {len(chunk.page_content)} caracteres")
    print(f"Metadata: {chunk.metadata}")
    print(f"Contenido: {chunk.page_content}")

--------------------------------------------------
Chunk 1: 463 caracteres
Metadata: {'source': 'Base_de_Conocimientos/BROCHURE Bootcamp AI ENGINEER FOR DEVELOPERS.pdf', 'page': 0}
Contenido: B o o t c a m p  
AI ENGINEER FORDEVELOPERS
Domina el desarrollo de soluciones inteligentes con LLMs,
RAG y agentes multiagente. Aprende a diseñar, integrar,
monitorear y desplegar arquitecturas de IA generativa
en entornos cloud, low-code y tiempo real.
HERRAMIENTAS
OpenAI Google AI Anthropic Meta AI Mistral AI Docker DeepSeek LangChain
LangGraph LlamaIndex CrewAI FastAPI Qdrant Supabase Firestore Evolution
API
Chatwoot Streamlit LangFuse LiveKit Deepgram V0
--------------------------------------------------
Chunk 2: 495 caracteres
Metadata: {'source': 'Base_de_Conocimientos/BROCHURE Bootcamp AI ENGINEER FOR DEVELOPERS.pdf', 'page': 1}
Contenido: ¿Sabías que hoy los desarrolladores pueden construir agentes inteligentes
capaces de razonar, colaborar entre sí y desplegarse en entornos
productivos c

## Paso 3 — Modelo de Embeddings

Un **embedding** es una representación numérica del texto en un espacio vectorial de alta dimensión, donde fragmentos semánticamente similares quedan geométricamente cerca entre sí. Esta representación es la que hace posible la **búsqueda por similitud semántica** en lugar de búsqueda por palabras clave exactas.

`text-embedding-ada-002` de OpenAI genera vectores de **1536 dimensiones**. Es importante que el mismo modelo se utilice tanto en esta fase (indexación) como en la Fase 2 (consulta), ya que la similitud entre vectores solo es significativa cuando fueron generados por el mismo espacio de representación.

In [9]:
#========== Paso 3: Embeddings - Cargamos el Modelo de Embeddings para convertir los Chunks ==========
embedding_model = OpenAIEmbeddings(
    model='text-embedding-ada-002' # otros modelos de vector embeddings: text-embedding-3-small, text-embedding-3-large
)

dim_model = len(embedding_model.embed_query("hola"))
print(f"El modelo de embeddings tiene dimensión: {dim_model}")

El modelo de embeddings tiene dimensión: 1536


## Paso 4 — Conexión a Supabase

[Supabase](https://supabase.com/) es una plataforma de base de datos PostgreSQL gestionada en la nube. Para el almacenamiento y búsqueda de vectores utiliza la extensión **`pgvector`**, que permite:

- Almacenar vectores de alta dimensión en columnas de PostgreSQL
- Ejecutar búsquedas de similitud coseno o por producto punto con índices eficientes (HNSW / IVFFlat)

Las credenciales se cargan desde variables de entorno (`.env`) para no exponer claves en el código:

| Variable | Descripción |
|----------|-------------|
| `SUPABASE_URL` | Endpoint REST de la instancia Supabase |
| `SUPABASE_SECRET_KEY` | Clave de servicio con permisos de lectura y escritura |

In [10]:
#======================= Paso 4: Carga de keys de supabase ====================

SUPABASE_URL = os.getenv("SUPABASE_URL")
SUPABASE_SECRET_KEY = os.getenv("SUPABASE_SECRET_KEY")

if not all([SUPABASE_URL, SUPABASE_SECRET_KEY]):
    raise ValueError(
        "❌ Faltan variables de base de datos en .env\n"
        "Requeridas: SUPABASE_URL, SUPABASE_SECRET_KEY"
    )

print(f"🔌 Listo para conectar a Supabase: {SUPABASE_URL}")

🔌 Listo para conectar a Supabase: https://ebdkewopehhsvkhboofb.supabase.co


In [11]:
# Nombre de la tabla en Supabase donde se almacenarán los vectores
# deben coincidir con el nombre de la tabla que se crea con la query del archivo "snippet_para_Supabase.sql"
TABLE_NAME_FOR_VECTORS = "documentos_vectorizados"

# Nombre de la función en Supabase que realizará la búsqueda de similitud
# deben coincidir con la función que se crea con la query del archivo "snippet_para_Supabase.sql"
FUNCTION_NAME_FOR_SIMILARITY_SEARCH = "similar_documentos_vectorizados"

## Paso 5 — Carga al VectorStore

`SupabaseVectorStore.from_documents()` encadena automáticamente dos operaciones:

1. **Generación de embeddings**: convierte cada chunk en un vector usando el modelo configurado
2. **Inserción en Supabase**: guarda en la tabla `documentos_vectorizados` el texto del chunk, su vector y los metadatos de origen

La función `similar_documentos_vectorizados` debe existir previamente en Supabase (creada con el SQL del archivo `snippet_para_Supabase.sql`). Esta función es la que se invocará en la **Fase 2** del pipeline RAG para recuperar los chunks más cercanos semánticamente a la consulta del usuario.

In [14]:
#======================= Paso 5: VectorStore - Llevamos los Embeddings a Supabase ====================

print(f"🔌 Conectando a Supabase...")
client = create_client(SUPABASE_URL, SUPABASE_SECRET_KEY)

# Truncar la tabla antes de cargar para evitar duplicados en re-ejecuciones
print(f"🗑️  Truncando la tabla '{TABLE_NAME_FOR_VECTORS}' antes de cargar nuevos embeddings...")
client.table(TABLE_NAME_FOR_VECTORS).delete().not_.is_("id", "null").execute()
print(f"✅ Tabla '{TABLE_NAME_FOR_VECTORS}' vaciada correctamente.")

print(f"\nEnviando los embeddings a la tabla '{TABLE_NAME_FOR_VECTORS}'...")

vectorstore = SupabaseVectorStore.from_documents(
    documents=chunks,
    embedding=embedding_model,
    client=client,
    table_name=TABLE_NAME_FOR_VECTORS, # Tabla en Supabase donde se almacenarán los vectores
    query_name=FUNCTION_NAME_FOR_SIMILARITY_SEARCH # Función en Supabase que realizará la búsqueda de similitud
)

print(f"✅ Embeddings enviados a Supabase y almacenados en la tabla '{TABLE_NAME_FOR_VECTORS}'")

🔌 Conectando a Supabase...
🗑️  Truncando la tabla 'documentos_vectorizados' antes de cargar nuevos embeddings...
✅ Tabla 'documentos_vectorizados' vaciada correctamente.

Enviando los embeddings a la tabla 'documentos_vectorizados'...
✅ Embeddings enviados a Supabase y almacenados en la tabla 'documentos_vectorizados'


## Resultado y Próximos Pasos

La **Fase 1** del pipeline RAG quedó completa. Los 46 chunks del PDF fueron vectorizados y almacenados en Supabase. Cada registro en la tabla `documentos_vectorizados` contiene:

| Campo | Descripción |
|-------|-------------|
| `content` | Texto del chunk |
| `embedding` | Vector de 1536 dimensiones |
| `metadata` | Página de origen, ruta del archivo |

### Fase 2 — Sistema de Consulta RAG (siguiente notebook)

Con la base de conocimientos lista, el siguiente paso es implementar el **retriever + LLM** que:

1. Convierte la pregunta del usuario en un embedding con el mismo modelo (`text-embedding-ada-002`)
2. Busca en Supabase los `k` chunks con mayor similitud coseno mediante `similar_documentos_vectorizados`
3. Inyecta esos chunks como contexto en el prompt del LLM
4. Genera una respuesta fundamentada en los documentos recuperados